# **1. 셀리니움**
셀레니움(Selenium)은 파이썬 코드로 실제 웹 브라우저(Chrome, Edge 등)를 자동으로 제어할 수 있게 해주는 웹 자동화 라이브러리입니다. 일반적인 requests 크롤링은 서버에서 받은 HTML만 분석하지만, Selenium은 브라우저를 직접 실행하여 버튼 클릭, 검색 입력, 스크롤, 로그인, 페이지 이동 등의 사용자 동작을 그대로 수행할 수 있기 때문에 JavaScript로 동적으로 생성되는 데이터까지 가져올 수 있습니다. 따라서 멜론, 유튜브, 쇼핑몰처럼 JavaScript 렌더링이 많은 사이트의 크롤링이나 웹 자동화 테스트에서 매우 많이 사용됩니다. 보통 webdriver.Chrome()으로 브라우저를 실행하고, find_element()로 요소를 찾으며, click(), send_keys() 등을 통해 자동화 작업을 수행합니다.

```
python -m pip install selenium
```

In [3]:
import requests
from bs4 import BeautifulSoup

url = "http://127.0.0.1:3000/workspace/python/4_teacher_bugs_crawling/7.html"    # 자바스크립트로 생성된 html이라서 정보를 가져올 수 없음..!! (동적)
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")
fruits = soup.select(".fruit")
print(fruits)

[]


In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

driver = webdriver.Chrome()     # selenium 객체 안에 Chrome() 메서드가 존재. 크롬브라우저를 컨트롤
url = "http://127.0.0.1:3000/workspace/python/4_teacher_bugs_crawling/7.html"
driver.get(url)     # 실행 시 크롬 창 띄워짐

# JavaScript 실행 대기
time.sleep(2)       # 2초 대기
fruits = driver.find_elements(By.CLASS_NAME, "fruit")       # 클래스 이름으로 찾겠다는 의미

for fruit in fruits:
    print(fruit.text)

driver.quit()

사과
바나나
오렌지
딸기


> 👉 find_elements()는 Selenium에서 HTML 요소를 여러 개 찾을 때 사용하는 메서드입니다. 하나의 요소만 찾는 find_element()와 달리, 조건에 맞는 요소들을 리스트 형태로 모두 반환합니다. 예를 들어 여러 개의 이미지, 게시글, 테이블 행(tr) 등을 반복해서 크롤링할 때 매우 자주 사용됩니다. 찾는 방식은 By.ID, By.CLASS_NAME, By.CSS_SELECTOR, By.XPATH 등 다양한 선택자를 사용할 수 있으며, 반환 결과는 리스트이므로 for문으로 반복 처리하는 경우가 많습니다. 또한 find_element()는 요소를 찾지 못하면 에러가 발생하지만, find_elements()는 빈 리스트([])를 반환하기 때문에 반복 크롤링에서 더 안정적으로 사용되는 경우도 많습니다.

***
# **2. 멜론 아티스트 곡 리스트 크롤링**

In [5]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:

def melon_search_from_main(keyword):
    options = Options()
    options.add_argument("--start-maximized")       # 브라우저 시작 시 브라우저 최대화 (반응형에서는 태그가 달라질 수 있기 때문에)

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10)        # 최대 10초 기다리는 시간 (조건을 만족하면 만족되는대로 넘어감)

    data = []

    try:
        # 웹 사이트 키기
        driver.get("https://www.melon.com/")
        time.sleep(2)

        search_box = wait.until(
            EC.presence_of_element_located((By.ID, "top_search"))   # ID가 top_search인 것을 찾아. 못찾으면 찾을 때까지 10초 대기. 나타나면 search_box에 대입
        )

        search_box.clear()
        search_box.send_keys(keyword)       # 검색어를 입력하고
        search_box.send_keys(Keys.ENTER)    # 엔터키를 입력하고
        time.sleep(3)       # 3초 대기

        song_tab = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, '//*[@id="divCollection"]/ul/li[3]/a/span')
            )
        )


        song_tab.click()

        time.sleep(3)

        song_table = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, '//*[@id="frm_defaultList"]/div/table')
            )
        )

        rows = song_table.find_elements(By.CSS_SELECTOR, "tbody tr")

        print("찾은 행 개수:", len(rows))

        for row in rows:
            try:
                cols = row.find_elements(By.TAG_NAME, "td")

                if len(cols) < 5:
                    continue

                # 곡명 td
                title_text = cols[2].text.strip()
                title_lines = [
                    line.strip()
                    for line in title_text.split("\n")
                    if line.strip()
                ]

                title = ""

                for line in title_lines:
                    if (
                        "재생" not in line
                        and "담기" not in line
                        and "상세정보" not in line
                        and not line.startswith("Title")
                    ):
                        title = line
                        break

                # Title 줄이 더 정확한 경우 보정
                for line in title_lines:
                    if line.startswith("Title "):
                        title = line.replace("Title ", "").strip()
                        break

                # 아티스트
                artist_text = cols[3].text.strip()
                artist_lines = [
                    line.strip()
                    for line in artist_text.split("\n")
                    if line.strip()
                ]
                artist = artist_lines[0] if artist_lines else ""

                # 앨범
                album_text = cols[4].text.strip()
                album_lines = [
                    line.strip()
                    for line in album_text.split("\n")
                    if line.strip()
                ]
                album = album_lines[0] if album_lines else ""

                # 좋아요 수
                like = ""
                try:
                    like = row.find_element(
                        By.CSS_SELECTOR,
                        "button.like span.cnt"
                    ).text.strip()
                except:
                    pass

                if title:
                    data.append({
                        "곡명": title,
                        "아티스트": artist,
                        "앨범": album,
                        "좋아요수": like
                    })

            except Exception as e:
                print("행 처리 오류:", e)

        df = pd.DataFrame(data)

        if not df.empty:
            df.index = df.index + 1

        file_name = f"melon_{keyword}_songs.csv"
        df.to_csv(file_name, encoding="utf-8-sig")

        print(f"CSV 저장 완료: {file_name}")
        print(f"총 {len(df)}곡 수집 완료")

        return df

    finally:
        driver.quit()
      

In [7]:

melon_search_from_main("신의키스")

찾은 행 개수: 6
CSV 저장 완료: melon_신의키스_songs.csv
총 6곡 수집 완료


,곡명,아티스트,앨범,좋아요수
1,이젠 내가 더 사랑해 (PJ1),신의키스,The Bucket List,5
2,헤어질 준비,신의키스,The Bucket List,5
3,너 없는 생일,신의키스,The Bucket List,4
4,버킷 리스트,신의키스,The Bucket List,5
5,서로를 가장 잘 아는 남,신의키스,The Bucket List,6
6,이젠 내가 더 사랑해 (PJ2),신의키스,The Bucket List,4


# ※ xpath
XPath는 XML 또는 HTML 문서 내에서 특정 요소나 속성을 선택하기 위해 사용되는 경로 표현 언어입니다. 웹 크롤링이나 자동화 도구에서 주로 사용되며, 요소를 효율적으로 찾을 수 있도록 도와줍니다. 일반적인 XPath는 특정 위치나 속성을 기준으로 요소를 선택하는 상대적인 경로를 사용합니다. 또한 full xpath는 루트 요소에서 시작하여 대상 요소까지의 절대적인 경로를 나타냅니다. 따라서 문서 구조가 변경되면 경로가 깨질 가능성이 높습니다.